# 04b — Phase 4A: which session statistics give the best live angle?

All three models are I_ML without look-ahead (200 trees × 31 leaves), **trained and tested
on streaming features** of the same variant, on people they never saw, on the same frames.

| name | bone lengths and scale come from | length unit |
|---|---|---|
| `window` (Phase 2) | the last 300 frames (~10 s); older frames are forgotten | shoulder width |
| `exp_sw` | **every frame since the session started** | shoulder width |
| `exp_torso` | every frame since the session started | **torso length** (mid-shoulder to mid-hip, as in `pose_normalise.py`); shoulder width until the hips have been seen |

No pose is asked of the patient: the session estimates simply keep improving as they exercise.

**Rule fixed in advance:** keep the variant with the lowest side-on (profile) error in E3;
if `exp_torso` is within 0.2° of the best, keep `exp_torso`, because it never forgets and
needs no pose from the patient.

In [ ]:
import os, sys, json, time, warnings
import numpy as np, pandas as pd
from scipy.stats import wilcoxon
import lightgbm as lgb

REPO  = r'F:\namal sir project\github repo\AI-based-Physiotherapy-project-'   # <-- your repo
FILES = {'rehab24': os.path.join(REPO, 'data', 'fit3d', 'feat_rehab24_stream.npz'),
         'fit3d':   os.path.join(REPO, 'data', 'fit3d', 'feat_fit3d_stream.npz')}
N_JOBS = max(1, (os.cpu_count() or 2) - 1)
sys.path[:0] = [REPO, os.path.join(REPO, 'scripts')]
from calibration import fit_curve, apply_calibration
from fit_calibration import extend_below
pd.set_option('display.width', 220)

def view_class(v):
    """camera-relative view (REHAB24-6 cam18 labels describe the subject's pose to cam17)"""
    v = str(v).lower(); cam18 = 'cam18' in v
    if 'half' in v: return 'half'
    if 'front' in v: return 'profile' if cam18 else 'front'
    if 'prof' in v or 'side' in v: return 'front' if cam18 else 'profile'
    return 'unknown'


V1 = {'rehab24': 'feat_rehab24_stream.npz', 'fit3d': 'feat_fit3d_stream.npz'}       # Phase 2
V2 = {'rehab24': 'feat_rehab24_stream_v2.npz', 'fit3d': 'feat_fit3d_stream_v2.npz'} # Phase 4A
VARS = {'window': 'X_stream', 'exp_sw': 'X_exp_sw', 'exp_torso': 'X_exp_torso'}
frames = []
for ds in V1:
    p1, p2 = (os.path.join(REPO, 'data', 'fit3d', v[ds]) for v in (V1, V2))
    if not (os.path.exists(p1) and os.path.exists(p2)):
        print('missing (skipped):', p1 if not os.path.exists(p1) else p2); continue
    d1, d2 = np.load(p1, allow_pickle=False), np.load(p2, allow_pickle=False)
    assert len(d1['y']) == len(d2['y']) and np.allclose(d1['y'], d2['y'], equal_nan=True) \
        and (d1['recording'] == d2['recording']).all(), f'{ds}: v1 and v2 files do not hold the same frames'
    names = [str(n) for n in d1['feature_names']]
    df = pd.concat([pd.DataFrame((d1 if v == 'X_stream' else d2)[v], columns=names).add_prefix(k + '_')
                    for k, v in VARS.items()], axis=1)
    for k in ('y', 'person', 'recording', 'view', 'exercise', 'dataset'):
        df[k] = d1[k]
    frames.append(df)
D = pd.concat(frames, ignore_index=True)
D['vclass'] = D['view'].map(view_class)
D = D[np.isfinite(D['window_angle_2d']) & np.isfinite(D['y'])].reset_index(drop=True)
DROP = {'bone_upper', 'bone_fore', 'sw_frac', 'is_right', 'std_angle', 'd_angle'}
FEATS = [n for n in names if n not in DROP]
print(D.groupby(['dataset', 'vclass']).agg(frames=('y', 'size'), people=('person', 'nunique')))

## How stable are the length features over a session?

In [ ]:
fps = np.where(D['dataset'] == 'fit3d', 25.0, 30.0)
D['t_s'] = D.groupby('recording').cumcount() / fps
band = pd.cut(D['t_s'], [0, 1, 3, 10, 1e9], labels=['0-1 s', '1-3 s', '3-10 s', '>10 s'], right=False)
# spread of the per-recording bone-length estimate: a good estimate settles and stays put
for v in VARS:
    cv = (D.groupby(['recording', band], observed=True)[v + '_plan_fore']
            .median().groupby(level=1, observed=True).std())
    print(f'{v:10s} spread of median forearm planarity by time:', cv.round(3).to_dict())

## Train and test every variant

In [ ]:
HP = dict(objective='l1', n_estimators=200, num_leaves=31, learning_rate=0.09, min_child_samples=80,
          subsample=0.8, subsample_freq=1, colsample_bytree=0.8, reg_lambda=1.0, verbosity=-1, n_jobs=N_JOBS)

def fit(tr, v):
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        return lgb.LGBMRegressor(**HP).fit(tr[[f'{v}_{f}' for f in FEATS]].to_numpy(), tr['y'])

def curve_pred(tr, te):
    def one(a, b):
        for mc in (300, 100, 30):
            try:
                return extend_below(fit_curve(a, b, min_count=mc))
            except ValueError:
                pass
    pooled = one(tr['window_angle_2d'], tr['y']); out = np.empty(len(te))
    for vc, idx in te.groupby('vclass').indices.items():
        sub = tr[tr['vclass'] == vc]
        p = one(sub['window_angle_2d'], sub['y']) if len(sub) >= 2000 else None
        out[idx] = apply_calibration(te['window_angle_2d'].to_numpy(float)[idx], p or pooled)
    return out

def run(train_mask, test_mask, label):
    tr, te = D[train_mask], D[test_mask].reset_index(drop=True)
    out = te[['dataset', 'person', 'recording', 'vclass', 'y']].copy()
    out['curve'] = curve_pred(tr, te)
    for v in VARS:
        out[v] = np.clip(fit(tr, v).predict(te[[f'{v}_{f}' for f in FEATS]].to_numpy()), 0, 180)
    out['experiment'] = label
    return out

t0 = time.time()
is_fit, is_reh = D['dataset'] == 'fit3d', D['dataset'] == 'rehab24'
parts = []
if is_fit.any() and is_reh.any():
    parts += [run(is_reh, is_fit, 'E2a REHAB->Fit3D'), run(is_fit, is_reh, 'E2b Fit3D->REHAB')]
for p in sorted(D['person'].unique()):
    parts.append(run(D['person'] != p, D['person'] == p, 'E3 LOSO'))
    print('  held out', p, f'{time.time() - t0:.0f}s', flush=True)
P = pd.concat(parts, ignore_index=True)

## Results

In [ ]:
M = ['curve'] + list(VARS)
err = P[M].sub(P['y'], axis=0).abs()
rec = pd.concat([P[['experiment', 'dataset', 'person', 'recording', 'vclass']], err], axis=1) \
        .groupby(['experiment', 'dataset', 'person', 'recording', 'vclass'], as_index=False)[M].mean()
tab = lambda r: r.groupby('experiment')[M].median().round(2).assign(recordings=r.groupby('experiment').size())
print('ALL VIEWS - median per-recording MAE (deg)'); display(tab(rec))
print('PROFILE VIEW (live setup)'); display(tab(rec[rec.vclass == 'profile']))

e3 = rec[rec.experiment == 'E3 LOSO']
rows = []
for scope, r in (('all views', e3), ('profile', e3[e3.vclass == 'profile'])):
    s = r.groupby('person')[M].mean()
    for a, b in (('window', 'exp_sw'), ('window', 'exp_torso'), ('curve', 'exp_torso')):
        g = s[a] - s[b]
        rows.append(dict(scope=scope, compare=f'{a} -> {b}', people=len(s), median_gain=round(float(g.median()), 2),
                         better_for=f'{int((g > 0).sum())}/{len(s)}', wilcoxon_p=round(float(wilcoxon(g).pvalue), 4)))
display(pd.DataFrame(rows))

In [ ]:
prof = e3[e3.vclass == 'profile'][list(VARS)].median()
best = prof.idxmin()
CHOICE = 'exp_torso' if prof['exp_torso'] - prof[best] <= 0.2 else best
print('E3 profile median error:', prof.round(2).to_dict())
print('CHOICE:', CHOICE)

## Retrain the deployable model with the chosen variant (REHAB24-6 only)

In [ ]:
reh = D[D['dataset'] == 'rehab24']
model = fit(reh, CHOICE)
mode = {'window': {'session': 'window', 'scale': 'shoulder'},
        'exp_sw': {'session': 'expanding', 'scale': 'shoulder'},
        'exp_torso': {'session': 'expanding', 'scale': 'torso'}}[CHOICE]
path = os.path.join(REPO, 'data', f'iml_nolook_rehab24_{CHOICE}.txt')
model.booster_.save_model(path)
json.dump({'features': FEATS, 'feature_stream': mode,          # IMLStream rebuilds the same features
           'params': {k: v for k, v in HP.items() if k != 'n_jobs'},
           'trained_on': 'REHAB24-6 only (CC BY-NC), streaming features',
           'cv': {'E3_all_views_deg': round(float(e3[CHOICE].median()), 2),
                  'E3_profile_deg': round(float(prof[CHOICE]), 2)}},
          open(path + '.json', 'w'), indent=1)
print('saved', path)
print(f'use it live:  python demo_cv.py --variant ML --iml-model data/iml_nolook_rehab24_{CHOICE}.txt')

In [ ]:
# optional: MLflow
try:
    import mlflow
    mlflow.set_tracking_uri('sqlite:///' + os.path.join(REPO, 'mlflow.db').replace('\\', '/'))
    mlflow.set_experiment('Physio-ROM-Estimation')
    with mlflow.start_run(run_name='phase4a-expanding-session-stats'):
        mlflow.set_tags({'stage': 'offline_evaluation', 'phase': '4A', 'choice': CHOICE})
        for scope, r in (('all', rec), ('profile', rec[rec.vclass == 'profile'])):
            for exp, row in tab(r).iterrows():
                for col in M:
                    mlflow.log_metric(f'{scope}_{exp.split()[0]}_{col}', float(row[col]))
        mlflow.log_artifact(path, artifact_path='model'); mlflow.log_artifact(path + '.json', artifact_path='model')
    print('logged to MLflow')
except ImportError:
    print('mlflow not installed - skipped')